# Library & Utils

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
import pandas as pd
import pytesseract
from pdf2image import convert_from_path
import re
from PIL import Image
from tqdm import tqdm
import torch 
from transformers import DonutProcessor, VisionEncoderDecoderModel
import numpy as np
import cv2
import keras_ocr
import tensorflow as tf
from paddleocr import PaddleOCR
from pdf2image import convert_from_path
import io
import base64
import requests
from openai import OpenAI
from difflib import get_close_matches

In [4]:
# Base Path Folder Proyek
BASE_DIR = os.getcwd() 

# 2. Set Lokasi Tesseract
path_tesseract = r"C:/Program Files/Tesseract-OCR/tesseract.exe"
if os.path.exists(path_tesseract):
    pytesseract.pytesseract.tesseract_cmd = path_tesseract
    print(f"✅ Tesseract ditemukan di: {path_tesseract}")
else:
    print("❌ Tesseract tidak ditemukan! Cek path-nya.")

✅ Tesseract ditemukan di: C:/Program Files/Tesseract-OCR/tesseract.exe


In [5]:
POPPLER_PATH = r"C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/bin/poppler-25.07.0/Library/bin"

# OCR

## Fungsi

In [6]:
print("Sedang memuat model Keras-OCR (TensorFlow)...")
# Pipeline ini akan mendownload model detektor & recognizer saat pertama kali dijalankan
pipeline = keras_ocr.pipeline.Pipeline()
print("✅ Model Keras-OCR siap digunakan.")

def run_keras_ocr(pil_image, pipeline):
    """
    Helper function untuk menjalankan Keras-OCR pada gambar PIL.
    Mengembalikan string teks hasil gabungan.
    """
    try:
        # 1. Konversi PIL ke Numpy Array (Keras-OCR butuh format ini)
        # Pastikan mode RGB
        img_array = np.array(pil_image.convert('RGB'))
        
        # 2. Jalankan Prediksi
        # pipeline.recognize menerima LIST gambar, jadi kita kasih [img_array]
        prediction_groups = pipeline.recognize([img_array])
        
        # 3. Ambil hasil untuk gambar pertama
        predictions = prediction_groups[0]
        
        # 4. Format output
        # Output Keras-OCR itu list of tuples: [('teks', box), ('teks', box)]
        # Kita cuma butuh teksnya, lalu digabung jadi satu string
        detected_text = " ".join([text for text, box in predictions])
        
        return detected_text
    except Exception as e:
        print(f"Error Keras-OCR: {e}")
        return ""

Sedang memuat model Keras-OCR (TensorFlow)...
Looking for C:\Users\ibuba\.keras-ocr\craft_mlt_25k.h5

Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.BILINEAR...)` instead.

Looking for C:\Users\ibuba\.keras-ocr\crnn_kurapan.h5
✅ Model Keras-OCR siap digunakan.


In [7]:
print("Sedang memuat model Donut...")
try:
    DONUT_MODEL_PATH = "naver-clova-ix/donut-base-finetuned-cord-v2"
    processor = DonutProcessor.from_pretrained(DONUT_MODEL_PATH)
    model = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_PATH)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    # Set model ke mode evaluasi biar hemat memori
    model.eval() 
    
    print(f"✅ Donut OCR Model berhasil diinisialisasi di: {device}")
    USE_DONUT = True
except Exception as e:
    print(f"❌ Gagal inisialisasi Donut: {e}")
    USE_DONUT = False

def run_donut_ocr(image_pil, processor, model, device):
    try:
        # Siapkan input
        pixel_values = processor(image_pil.convert("RGB"), return_tensors="pt").pixel_values
        
        # Siapkan prompt decoder (untuk OCR task)
        task_prompt = "<s_ocr>"
        decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
        
        # Generate output
        with torch.no_grad(): # Matikan gradien biar cepet & hemat memori
            outputs = model.generate(
                pixel_values.to(device),
                decoder_input_ids=decoder_input_ids.to(device),
                max_length=768, # Sesuaikan panjang maks teks
                early_stopping=True,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
                num_beams=1, # Gunakan 1 beam biar lebih cepat, 4 kalau mau lebih akurat tapi lambat
                bad_words_ids=[[processor.tokenizer.unk_token_id]],
                return_dict_in_generate=True,
            )
        
        # Decode hasil
        sequence = processor.batch_decode(outputs.sequences)[0]
        sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
        
        # Bersihkan tag spesial Donut
        cleaned_text = re.sub(r"<.*?>", "", sequence).strip()
        
        return cleaned_text
    except Exception as e:
        print(f"Error Donut: {e}")
        return ""

Sedang memuat model Donut...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Donut OCR Model berhasil diinisialisasi di: cpu


In [8]:
print("Sedang memuat model PaddleOCR...")
# use_angle_cls=True --> Mengaktifkan deteksi orientasi di level model
ocr_engine = PaddleOCR(use_angle_cls=True, lang='en')
print("✅ Model PaddleOCR siap digunakan.")

def run_paddle_ocr(pil_image, engine):
    """
    Helper function untuk menjalankan PaddleOCR pada gambar PIL.
    """
    try:
        # 1. Konversi PIL ke Numpy Array (RGB)
        img_array = np.array(pil_image.convert('RGB'))
        
        # 2. Jalankan Prediksi (FIX: Hapus cls=True)
        # Orientasi sudah dihandle oleh setting 'use_angle_cls=True' di init engine
        result = engine.ocr(img_array)
        
        # 3. Parsing Hasil
        # Paddle balikin list of lists. Kalau kosong (gak ada teks), result-nya [None] atau None
        if result is None or len(result) == 0 or result[0] is None:
            return ""
            
        # Struktur result v2: [[[[coords], [text, conf]], ...]]
        # Kita cuma butuh text-nya
        detected_texts = [line[1][0] for line in result[0]]
        
        # Gabung jadi satu string
        full_text = " ".join(detected_texts)
        
        return full_text
    except Exception as e:
        # Print error spesifik untuk debugging tapi return string kosong biar loop lanjut
        print(f"Warning PaddleOCR: {e}") 
        return ""

Sedang memuat model PaddleOCR...


C:\Users\ibuba\AppData\Local\Temp\ipykernel_32332\1773259101.py:3: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr_engine = PaddleOCR(use_angle_cls=True, lang='en')
c:\Users\ibuba\Kuliah\Proyek Akhir\Aplikasi\venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ibuba\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ibuba\.paddlex\offic

✅ Model PaddleOCR siap digunakan.


In [9]:
# 1. KONFIGURASI LM STUDIO (SERVER)
LM_STUDIO_URL = "http://127.0.0.1:1234/v1"
API_KEY = "qwenllmforocronlywiththehighestlengthofcodegeneration"
MODEL_ID = "qwen/qwen3-vl-4b" # Pastikan sama dengan di LM Studio lo

print(f"Menghubungkan ke LM Studio ({MODEL_ID})...")
try:
    client = OpenAI(base_url=LM_STUDIO_URL, api_key=API_KEY)
    client.models.list() # Cek koneksi
    print("✅ Koneksi LM Studio BERHASIL.")
except Exception as e:
    print(f"❌ Koneksi GAGAL: {e}")
    print("Pastikan Server LM Studio sudah START.")
    exit() # Stop script kalau server mati

Menghubungkan ke LM Studio (qwen/qwen3-vl-4b)...
✅ Koneksi LM Studio BERHASIL.


In [10]:
# FUNGSI OCR VIA LM STUDIO
def encode_image_to_base64(pil_image):
    """Ubah gambar jadi string Base64."""
    buffered = io.BytesIO()
    pil_image.convert("RGB").save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def run_lmstudio_ocr(pil_image):
    """
    Mengirim gambar ke Qwen-VL di LM Studio buat OCR.
    """
    try:
        base64_image = encode_image_to_base64(pil_image)
        
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text", 
                            "text": "OCR Task: Extract ALL text from this image exactly as it appears. Output ONLY the text, no conversational filler."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                }
            ],
            temperature=0.1, 
            max_tokens=8192, 
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ Error Request: {e}")
        return ""

In [11]:
# 1. FUNGSI AUGMENTASI
def apply_augmentations(pil_img):
    augmented_images = {}
    img_cv = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)

    # Augmentasi 1: Rotasi Ringan
    M_rot = cv2.getRotationMatrix2D((img_cv.shape[1]/2, img_cv.shape[0]/2), 2, 1)
    img_rot = cv2.warpAffine(img_cv, M_rot, (img_cv.shape[1], img_cv.shape[0]), borderValue=(255,255,255))
    augmented_images["aug_rotate_2"] = Image.fromarray(cv2.cvtColor(img_rot, cv2.COLOR_BGR2RGB))

    # Augmentasi 2: Blur
    img_blur = cv2.GaussianBlur(img_cv, (3, 3), 0)
    augmented_images["aug_blur_3"] = Image.fromarray(cv2.cvtColor(img_blur, cv2.COLOR_BGR2RGB))

    # Augmentasi 3: Noise
    noise = np.zeros(img_cv.shape, np.uint8)
    cv2.randn(noise, 0, 20) 
    img_noise = cv2.add(img_cv, noise)
    augmented_images["aug_noise_20"] = Image.fromarray(cv2.cvtColor(img_noise, cv2.COLOR_BGR2RGB))

    return augmented_images

## Pytesseract

In [11]:
ijazah = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah" 
sertifikat = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikat"
folders_to_process = [ijazah, sertifikat]
ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')

# --- Tentukan folder output untuk gambar augmentasi ---
AUG_OUTPUT_DIR = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/" 
os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)
print(f"File augmentasi akan disimpan di: {AUG_OUTPUT_DIR}")

for folder_path in folders_to_process:
    
    # --- PERUBAHAN 1: Tentukan Jenis Dokumen ---
    jenis_dokumen = 'unknown' # Default
    if folder_path == ijazah:
        jenis_dokumen = 'ijazah'
    elif folder_path == sertifikat:
        jenis_dokumen = 'sertifikat'
    # ---------------------------------------------

    print(f"\n===== Memulai proses di folder: {folder_path} (Jenis: {jenis_dokumen}) =====")
    
    if not os.path.exists(folder_path) or not os.listdir(folder_path):
        print("Folder ini kosong atau tidak ditemukan, lanjut ke folder berikutnya.")
        continue

    for filename in tqdm(os.listdir(folder_path), desc=f"Memproses {os.path.basename(folder_path)}"):
        if filename.lower().endswith(allowed_extensions):
            file_path = os.path.join(folder_path, filename)
            try:
                image = None
                
                if filename.lower().endswith('.pdf'):
                    images_from_pdf = convert_from_path(
                        file_path, first_page=1, last_page=1, poppler_path=POPPLER_PATH
                    )
                    if images_from_pdf:
                        image = images_from_pdf[0]
                else:
                    image = Image.open(file_path)
                    
                if image:
                    # --- PROSES 1: OCR GAMBAR ASLI (GROUND TRUTH) ---
                    raw_text_original = pytesseract.image_to_string(image, lang='ind+eng')
                    lines = raw_text_original.split('\n')
                    cleaned_lines = [line.strip() for line in lines if line.strip()]
                    single_line_text_original = ' '.join(cleaned_lines)
                    
                    # Simpan hasil ASLI
                    ocr_results.append({
                        'nama_file_sumber': filename,
                        'nama_file_output': filename,
                        'hasil_ocr': single_line_text_original,
                        'augmentasi': 'original',
                        'jenis': jenis_dokumen 
                    })
                    
                    # --- PROSES 2: AUGMENTASI & OCR BARU ---
                    base_name, _ = os.path.splitext(filename)
                    augmented_images = apply_augmentations(image)

                    for aug_name, aug_img in augmented_images.items():
                        
                        aug_filename = f"{base_name}_{aug_name}.jpg"
                        aug_filepath = os.path.join(AUG_OUTPUT_DIR, aug_filename)
                        
                        aug_img.save(aug_filepath, "JPEG")
                        raw_text_aug = pytesseract.image_to_string(aug_img, lang='ind+eng')
                        lines_aug = raw_text_aug.split('\n')
                        cleaned_lines_aug = [line.strip() for line in lines_aug if line.strip()]
                        single_line_text_aug = ' '.join(cleaned_lines_aug)
                        
                        ocr_results.append({
                            'nama_file_sumber': filename, 
                            'nama_file_output': aug_filename, 
                            'hasil_ocr': single_line_text_aug, 
                            'augmentasi': aug_name,
                            'jenis': jenis_dokumen # <-- PERUBAHAN 3
                        })

            except Exception as e:
                tqdm.write(f"\n!!! Gagal memproses file {filename}: {e} !!!")
                ocr_results.append({
                    'nama_file_sumber': filename,
                    'nama_file_output': 'ERROR',
                    'hasil_ocr': f'ERROR: {e}',
                    'augmentasi': 'ERROR',
                    'jenis': jenis_dokumen
                })

# Cek jika ada hasil untuk disimpan
if ocr_results:
    df = pd.DataFrame(ocr_results)
    output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_donut_pytesseract.csv'
    df.to_csv(output_csv_path, index=False)
    print(f"Hasil OCR berhasil disimpan di: {output_csv_path}")
    print("\nPreview Hasil:")
    display(df.head())
else:
    print("Tidak ada hasil OCR untuk disimpan.")

File augmentasi akan disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/

===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah (Jenis: ijazah) =====


Memproses ijazah: 100%|██████████| 90/90 [04:41<00:00,  3.12s/it]



===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikat (Jenis: sertifikat) =====


Memproses sertifikat:  48%|████▊     | 27/56 [24:25<09:36, 19.90s/it]   


!!! Gagal memproses file IWD_2024_Dwijo_Utomo_Rahino_Putro.pdf: Image size (316394400 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack. !!!


Memproses sertifikat: 100%|██████████| 56/56 [29:09<00:00, 31.25s/it]


Hasil OCR berhasil disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_donut_pytesseract.csv

Preview Hasil:


,nama_file_sumber,nama_file_output,hasil_ocr,augmentasi,jenis
0,02081e1b-833a-494d-8e71-992671938014-160213173...,02081e1b-833a-494d-8e71-992671938014-160213173...,Nomor Seri : 0512/61201/2612 UN43. 001- 002038...,original,ijazah
1,02081e1b-833a-494d-8e71-992671938014-160213173...,02081e1b-833a-494d-8e71-992671938014-160213173...,UN43. 001- 002038 Nomor Seri : 0512/61 2061/2 ...,aug_rotate_2,ijazah
2,02081e1b-833a-494d-8e71-992671938014-160213173...,02081e1b-833a-494d-8e71-992671938014-160213173...,Nomor Seri : 0512/61201/2012 UN43. 001- 002038...,aug_blur_3,ijazah
3,02081e1b-833a-494d-8e71-992671938014-160213173...,02081e1b-833a-494d-8e71-992671938014-160213173...,Nomor Seri : 0512/61201/2612 UN43. 001- 002038...,aug_noise_20,ijazah
4,02081e1b-833a-494d-8e71-992671938014-160213173...,02081e1b-833a-494d-8e71-992671938014-160213173...,Nomor Seri : 0512/61201/2612 UN43. 001- 002038...,original,ijazah


## Donut

In [ ]:
ijazah = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah"
sertifikat = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikat"
folders_to_process = [ijazah, sertifikat]

# Setup Output
ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')
AUG_OUTPUT_DIR = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data_donut/"
os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)
print(f"File augmentasi akan disimpan di: {AUG_OUTPUT_DIR}")

if not USE_DONUT:
    print("❌ STOP: Model Donut gagal dimuat. Cek instalasi library.")
else:
    for folder_path in folders_to_process:
        
        # --- LOGIC JENIS DOKUMEN ---
        jenis_dokumen = 'unknown'
        if folder_path == ijazah:
            jenis_dokumen = 'ijazah'
        elif folder_path == sertifikat:
            jenis_dokumen = 'sertifikat'

        print(f"\n===== Memulai proses di folder: {folder_path} (Jenis: {jenis_dokumen}) =====")
        
        if not os.path.exists(folder_path) or not os.listdir(folder_path):
            print("Folder kosong/tidak ditemukan.")
            continue

        for filename in tqdm(os.listdir(folder_path), desc=f"Processing {os.path.basename(folder_path)}"):
            if filename.lower().endswith(allowed_extensions):
                file_path = os.path.join(folder_path, filename)
                try:
                    image = None
                    
                    # --- HANDLING PDF/IMAGE ---
                    if filename.lower().endswith('.pdf'):
                        # Convert PDF dengan resolusi menengah (200 DPI cukup buat Donut)
                        images_from_pdf = convert_from_path(
                            file_path, first_page=1, last_page=1, dpi=200, poppler_path=POPPLER_PATH
                        )
                        if images_from_pdf: 
                            image = images_from_pdf[0]
                    else:
                        image = Image.open(file_path)
                    
                    # --- VALIDASI & RESIZE ---
                    if image:
                        # Donut bisa lambat kalau gambar kegedean (misal 4000px). Resize kalau perlu.
                        w, h = image.size
                        if w > 2500 or h > 2500:
                            ratio = 2048 / max(w, h)
                            new_size = (int(w * ratio), int(h * ratio))
                            image = image.resize(new_size, Image.LANCZOS)
                        
                        # --- PROSES 1: OCR ORIGINAL (DONUT) ---
                        text_original = run_donut_ocr(image, processor, model, device)
                        
                        # Cleaning ringan (spasi ganda)
                        text_original = re.sub(r'\s+', ' ', text_original).strip()
                        
                        ocr_results.append({
                            'nama_file_sumber': filename,
                            'nama_file_output': filename,
                            'hasil_ocr': text_original,
                            'augmentasi': 'original',
                            'jenis': jenis_dokumen 
                        })
                        
                        # --- PROSES 2: AUGMENTASI & OCR BARU (DONUT) ---
                        base_name, _ = os.path.splitext(filename)
                        augmented_images = apply_augmentations(image)

                        for aug_name, aug_img in augmented_images.items():
                            aug_filename = f"{base_name}_{aug_name}.jpg"
                            aug_filepath = os.path.join(AUG_OUTPUT_DIR, aug_filename)
                            aug_img.save(aug_filepath, "JPEG")
                            
                            # Jalankan Donut pada gambar augmentasi
                            text_aug = run_donut_ocr(aug_img, processor, model, device)
                            text_aug = re.sub(r'\s+', ' ', text_aug).strip()
                            
                            ocr_results.append({
                                'nama_file_sumber': filename, 
                                'nama_file_output': aug_filename, 
                                'hasil_ocr': text_aug, 
                                'augmentasi': aug_name,
                                'jenis': jenis_dokumen
                            })

                except Exception as e:
                    tqdm.write(f"!!! Gagal memproses file {filename}: {e} !!!")
                    ocr_results.append({
                        'nama_file_sumber': filename,
                        'nama_file_output': 'ERROR',
                        'hasil_ocr': f'ERROR: {e}',
                        'augmentasi': 'ERROR',
                        'jenis': jenis_dokumen
                    })
                finally:
                    # Bersihkan cache GPU tiap iterasi biar gak OOM (Out of Memory)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

    # Simpan Hasil
    if ocr_results:
        df = pd.DataFrame(ocr_results)
        output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_donut_augmented.csv'
        df.to_csv(output_csv_path, index=False)
        print(f"\nHasil OCR (Donut) disimpan di: {output_csv_path}")
        print(df.head())
    else:
        print("Tidak ada data yang diproses.")

File augmentasi akan disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data_donut/

===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah (Jenis: ijazah) =====


Processing ijazah: 100%|██████████| 90/90 [3:44:16<00:00, 149.52s/it]  



===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikat (Jenis: sertifikat) =====


Processing sertifikat:  48%|████▊     | 27/56 [48:23<39:57, 82.67s/it]   

!!! Gagal memproses file IWD_2024_Dwijo_Utomo_Rahino_Putro.pdf: Image size (316394400 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack. !!!


Processing sertifikat: 100%|██████████| 56/56 [1:41:43<00:00, 108.98s/it]



Hasil OCR (Donut) disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_donut_augmented.csv
                                    nama_file_sumber  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  02081e1b-833a-494d-8e71-992671938014-160213173...   

                                    nama_file_output  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  02081e1b-833a-494d-8e71-992671938014-160213173...   

                                           hasil_ocr    augmentasi   jenis  
0  DAN DAN DAN KEBUDAYAAN UN43 001-00203818181818...      original  ijazah  
1  DAN DAN DAN N N N N N N N N N N N N N N N N N ...  aug_

: 

#### hidden cell

In [3]:
try:
    DONUT_MODEL_PATH = "naver-clova-ix/donut-base-finetuned-cord-v2"
    processor = DonutProcessor.from_pretrained(DONUT_MODEL_PATH)
    model = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_PATH)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    print("✅ Donut OCR Model berhasil diinisialisasi dan menggunakan:", device)
    USE_DONUT = True
except Exception as e:
    print(f"❌ Gagal inisialisasi Donut. Menggunakan Tesseract sebagai fallback. Error: {e}")
    USE_DONUT = False

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Donut OCR Model berhasil diinisialisasi dan menggunakan: cpu


In [ ]:
image_path = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/Support Document/sertifikat"

# List semua folder yang mau diproses
folders_to_process = [image_path]
ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')
global_file_id = 1

for folder_path in folders_to_process:
    print(f"\n===== Memulai proses di folder: {folder_path} =====")
    
    if not os.path.exists(folder_path):
        print(f"Folder TIDAK DITEMUKAN: {folder_path}. Lanjut ke folder berikutnya.")
        continue

    if not os.listdir(folder_path):
        print("Folder ini kosong, lanjut ke folder berikutnya.")
        continue

    all_files = os.listdir(folder_path)

for filename in tqdm(all_files, desc=f"Memproses {os.path.basename(folder_path)}"):
    if filename.lower().endswith(allowed_extensions):
        file_path = os.path.join(folder_path, filename)
        image = None
        
        try:
            # ========== KONVERSI PDF ==========
            if filename.lower().endswith('.pdf'):
                tqdm.write(f"\n[PDF] Processing: {filename}")
                
                try:
                    images_from_pdf = convert_from_path(
                        file_path,
                        first_page=1,
                        last_page=1,
                        dpi=200,
                        poppler_path=POPPLER_PATH,
                        fmt='jpeg',
                        thread_count=1
                    )
                    
                    if images_from_pdf and len(images_from_pdf) > 0:
                        image = images_from_pdf[0]
                        # PENTING: Load image data immediately
                        image.load()
                        tqdm.write(f"    [OK] PDF converted at 200 DPI")
                    else:
                        raise ValueError("PDF conversion returned empty list")
                        
                except Exception as e_pdf:
                    raise ValueError(f"PDF conversion error: {str(e_pdf)}")
            
            # ========== BUKA GAMBAR ==========
            else:
                tqdm.write(f"\n[IMAGE] Processing: {filename}")
                image = Image.open(file_path)
                # PENTING: Load image data
                image.load()
            
            # ========== VALIDASI GAMBAR ==========
            if image is None:
                raise ValueError("Image is None after loading")
            
            width, height = image.size
            tqdm.write(f"    Image info: {width}x{height}, mode={image.mode}, format={getattr(image, 'format', 'Unknown')}")
            
            if width == 0 or height == 0:
                raise ValueError(f"Invalid dimensions: {width}x{height}")
            
            # Resize jika terlalu besar (lakukan di sini, bukan di fungsi OCR)
            if width > 2500 or height > 2500:
                tqdm.write(f"    [INFO] Large image, resizing...")
                ratio = 2048 / max(width, height)
                new_size = (int(width * ratio), int(height * ratio))
                image = image.resize(new_size, Image.LANCZOS)
                image.load()  # Load ulang setelah resize
                tqdm.write(f"    [INFO] Resized to {image.size}")
            
            # ========== PROSES OCR ==========
            if USE_DONUT:
                try:
                    raw_text = run_donut_ocr(image, processor, model, device)
                    method_name = "DONUT"
                except Exception as e_donut:
                    tqdm.write(f"    [FALLBACK] Donut failed: {str(e_donut)}")
                    tqdm.write(f"    [INFO] Trying Tesseract...")
                    
                    # Fallback ke Tesseract
                    try:
                        preprocessed_image = preprocess_pil_image(image)
                        raw_text = pytesseract.image_to_string(preprocessed_image, lang='ind+eng')
                        method_name = "TESSERACT_FALLBACK"
                    except Exception as e_tess:
                        raise Exception(f"Both OCR methods failed. Donut: {str(e_donut)}, Tesseract: {str(e_tess)}")
            else:
                preprocessed_image = preprocess_pil_image(image)
                raw_text = pytesseract.image_to_string(preprocessed_image, lang='ind+eng')
                method_name = "TESSERACT"
            
            # Simpan hasil
            row = {
                'id': global_file_id,
                'nama_file': filename,
                'halaman': 1,
                'metode_ocr': method_name,
            }
            ocr_results.append(row)
            global_file_id += 1
            
            tqdm.write(f"✓ SUCCESS: {filename} ({method_name})")
            
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            tqdm.write(f"✗ ERROR: {filename} - {error_msg}")
            
            ocr_results.append({
                'id': global_file_id,
                'nama_file': filename,
                'halaman': 0,
                'metode_ocr': "ERROR",
                'hasil_ocr': error_msg
            })
            global_file_id += 1
            continue
        
        finally:
            # Cleanup memory
            if image is not None:
                try:
                    image.close()
                except:
                    pass
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# Simpan hasil ke CSV
output_csv = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/Support Document/hasil_ocr_sertifikat.csv'
df_results = pd.DataFrame(ocr_results)
df_results.to_csv(output_csv, index=False)

print(f"\n\n✅ Selesai! Total {len(df_results)} baris data diekstrak dan disimpan di {output_csv}")


===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/Support Document/sertifikat =====


Memproses sertifikat:   0%|          | 0/289 [00:00<?, ?it/s]


[IMAGE] Processing: 0_SEDytpvBslqw4TNp_jpg.rf.04898ca1b39a626f6913397aba189b16.jpg
    Image info: 640x640, mode=RGB, format=JPEG
    [DEBUG] Image size: (640, 640), mode: RGB
    [INFO] Processing with Donut processor...


Memproses sertifikat:   0%|          | 0/289 [00:00<?, ?it/s]

    [DEBUG] Pixel values shape: torch.Size([1, 3, 1280, 960])
    [INFO] Generating text with Donut model...


...

    [SUCCESS] Generated 1027 characters
✓ SUCCESS: TRIZ-1024x724_jpg.rf.f52fcdf3e4a25e16cf7f18797cabfbb4.jpg (DONUT)
   Result preview: <s_ocr> of Technology nnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnn...

[IMAGE] Processing: TT-MTP-O_jpg.rf.37c05aff9dfb53a82dc78260d56f685a.jpg
    Image info: 640x640, mode=RGB, format=JPEG
    [DEBUG] Image size: (640, 640), mode: RGB
    [INFO] Processing with Donut processor...
    [DEBUG] Pixel values shape: torch.Size([1, 3, 1280, 960])
    [INFO] Generating text with Donut model...


Memproses sertifikat: 100%|█████████▉| 288/289 [3:04:55<00:41, 41.44s/it]

    [SUCCESS] Generated 512 characters
✓ SUCCESS: TT-MTP-O_jpg.rf.37c05aff9dfb53a82dc78260d56f685a.jpg (DONUT)
   Result preview: <s_ocr>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>...

[IMAGE] Processing: TT-MTP-O_jpg.rf.5639dadc86909aa4737f8c9872807d79.jpg
    Image info: 640x640, mode=RGB, format=JPEG
    [DEBUG] Image size: (640, 640), mode: RGB
    [INFO] Processing with Donut processor...
    [DEBUG] Pixel values shape: torch.Size([1, 3, 1280, 960])
    [INFO] Generating text with Donut model...


Memproses sertifikat: 100%|██████████| 289/289 [3:05:35<00:00, 38.53s/it]


    [SUCCESS] Generated 512 characters
✓ SUCCESS: TT-MTP-O_jpg.rf.5639dadc86909aa4737f8c9872807d79.jpg (DONUT)
   Result preview: <s_ocr>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>...


✅ Selesai! Total 289 baris data diekstrak dan disimpan di C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/Support Document/hasil_ocr_sertifikat.csv


## Keras - OCR

In [16]:
# Ganti path sesuai lokal lo
ijazah = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah" 
sertifikat = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikat"

folders_to_process = [ijazah, sertifikat]
ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')

AUG_OUTPUT_DIR = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/"
os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)
print(f"File augmentasi akan disimpan di: {AUG_OUTPUT_DIR}")

for folder_path in folders_to_process:
    
    # Tentukan Jenis Dokumen
    jenis_dokumen = 'unknown'
    if folder_path == ijazah:
        jenis_dokumen = 'ijazah'
    elif folder_path == sertifikat:
        jenis_dokumen = 'sertifikat'

    print(f"\n===== Memulai proses di folder: {folder_path} (Jenis: {jenis_dokumen}) =====")
    
    if not os.path.exists(folder_path) or not os.listdir(folder_path):
        print("Folder kosong/tidak ditemukan.")
        continue

    for filename in tqdm(os.listdir(folder_path), desc=f"Memproses {os.path.basename(folder_path)}"):
        if filename.lower().endswith(allowed_extensions):
            file_path = os.path.join(folder_path, filename)
            try:
                image = None
                if filename.lower().endswith('.pdf'):
                    images_from_pdf = convert_from_path(
                        file_path, first_page=1, last_page=1,
                        poppler_path=POPPLER_PATH
                    )
                    if images_from_pdf: image = images_from_pdf[0]
                else:
                    image = Image.open(file_path)
                    
                if image:
                    # --- PROSES 1: OCR GAMBAR ASLI (KERAS-OCR) ---
                    # Panggil helper function Keras-OCR
                    text_original = run_keras_ocr(image, pipeline)
                    
                    # Cleaning ringan (spasi ganda)
                    text_original = re.sub(r'\s+', ' ', text_original).strip()
                    
                    ocr_results.append({
                        'nama_file_sumber': filename,
                        'nama_file_output': filename,
                        'hasil_ocr': text_original,
                        'augmentasi': 'original',
                        'jenis': jenis_dokumen 
                    })
                    
                    # --- PROSES 2: AUGMENTASI & OCR BARU ---
                    base_name, _ = os.path.splitext(filename)
                    augmented_images = apply_augmentations(image)

                    for aug_name, aug_img in augmented_images.items():
                        aug_filename = f"{base_name}_{aug_name}.jpg"
                        aug_filepath = os.path.join(AUG_OUTPUT_DIR, aug_filename)
                        aug_img.save(aug_filepath, "JPEG")
                        
                        # Jalankan Keras-OCR pada gambar augmentasi
                        text_aug = run_keras_ocr(aug_img, pipeline)
                        text_aug = re.sub(r'\s+', ' ', text_aug).strip()
                        
                        ocr_results.append({
                            'nama_file_sumber': filename, 
                            'nama_file_output': aug_filename, 
                            'hasil_ocr': text_aug, 
                            'augmentasi': aug_name,
                            'jenis': jenis_dokumen
                        })

            except Exception as e:
                tqdm.write(f"\n!!! Gagal memproses file {filename}: {e} !!!")
                ocr_results.append({
                    'nama_file_sumber': filename,
                    'nama_file_output': 'ERROR',
                    'hasil_ocr': f'ERROR: {e}',
                    'augmentasi': 'ERROR',
                    'jenis': jenis_dokumen
                })

# Simpan Hasil
if ocr_results:
    df = pd.DataFrame(ocr_results)
    output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/dataset/hasil_ocr_keras.csv'
    df.to_csv(output_csv_path, index=False)
    print(f"\nHasil OCR (Keras-OCR) disimpan di: {output_csv_path}")
    print(df.head())

File augmentasi akan disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/

===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazah (Jenis: ijazah) =====


Memproses ijazah:   0%|          | 0/90 [00:00<?, ?it/s]

5/5 [==============================] - 6s 1s/step


Memproses ijazah:   1%|          | 1/90 [01:00<1:29:39, 60.44s/it]

5/5 [==============================] - 6s 1s/step


...

2/2 [==============================] - 3s 1s/step


Memproses sertifikat:  98%|█████████▊| 55/56 [36:15<00:37, 37.57s/it]

3/3 [==============================] - 3s 729ms/step


Memproses sertifikat: 100%|██████████| 56/56 [36:57<00:00, 39.60s/it]



Hasil OCR (Keras-OCR) disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/dataset/hasil_ocr_keras.csv
                                    nama_file_sumber  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  02081e1b-833a-494d-8e71-992671938014-160213173...   

                                    nama_file_output  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  02081e1b-833a-494d-8e71-992671938014-160213173...   

                                           hasil_ocr    augmentasi   jenis  
0  unas ool 002038 seri 0512161201 zo1z omor dan ...      original  ijazah  
1  002038 ool una3 kebudayaan 2012 os12161201 dan...  aug_rotate

## PaddleOCR

In [7]:
import os
# Matikan log debug Paddle yang berisik
os.environ['FLAGS_allocator_strategy'] = 'auto_growth'
import logging
# Suppress paddle logs
logging.getLogger("ppocr").setLevel(logging.ERROR)

ijazah = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazahllm" 
sertifikat = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikatllm"
folders_to_process = [ijazah, sertifikat]

ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')

AUG_OUTPUT_DIR = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/"
os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)
print(f"File augmentasi akan disimpan di: {AUG_OUTPUT_DIR}")

for folder_path in folders_to_process:
    
    # --- LOGIC JENIS DOKUMEN ---
    jenis_dokumen = 'unknown'
    if folder_path == ijazah:
        jenis_dokumen = 'ijazah'
    elif folder_path == sertifikat:
        jenis_dokumen = 'sertifikat'

    print(f"\n===== Memulai proses di folder: {folder_path} (Jenis: {jenis_dokumen}) =====")
    
    if not os.path.exists(folder_path) or not os.listdir(folder_path):
        print("Folder kosong/tidak ditemukan.")
        continue

    for filename in tqdm(os.listdir(folder_path), desc=f"Memproses {os.path.basename(folder_path)}"):
        if filename.lower().endswith(allowed_extensions):
            file_path = os.path.join(folder_path, filename)
            try:
                image = None
                if filename.lower().endswith('.pdf'):
                    # Convert PDF
                    # poppler_path=r"C:\..." jika di windows lokal
                    images_from_pdf = convert_from_path(file_path, first_page=1, last_page=1, poppler_path=POPPLER_PATH)
                    if images_from_pdf: image = images_from_pdf[0]
                else:
                    image = Image.open(file_path)
                    
                if image:
                    # --- PROSES 1: OCR GAMBAR ASLI (PADDLE) ---
                    text_original = run_paddle_ocr(image, ocr_engine)
                    text_original = re.sub(r'\s+', ' ', text_original).strip()
                    
                    ocr_results.append({
                        'nama_file_sumber': filename,
                        'nama_file_output': filename,
                        'hasil_ocr': text_original,
                        'augmentasi': 'original',
                        'jenis': jenis_dokumen 
                    })
                    
                    # --- PROSES 2: AUGMENTASI & OCR BARU ---
                    base_name, _ = os.path.splitext(filename)
                    augmented_images = apply_augmentations(image)

                    for aug_name, aug_img in augmented_images.items():
                        aug_filename = f"{base_name}_{aug_name}.jpg"
                        aug_filepath = os.path.join(AUG_OUTPUT_DIR, aug_filename)
                        aug_img.save(aug_filepath, "JPEG")
                        
                        text_aug = run_paddle_ocr(aug_img, ocr_engine)
                        text_aug = re.sub(r'\s+', ' ', text_aug).strip()
                        
                        ocr_results.append({
                            'nama_file_sumber': filename, 
                            'nama_file_output': aug_filename, 
                            'hasil_ocr': text_aug, 
                            'augmentasi': aug_name,
                            'jenis': jenis_dokumen
                        })

            except Exception as e:
                # print error tapi jangan stop loop
                tqdm.write(f"!!! Gagal memproses file {filename}: {e} !!!")
                ocr_results.append({
                    'nama_file_sumber': filename,
                    'nama_file_output': 'ERROR',
                    'hasil_ocr': f'ERROR: {e}',
                    'augmentasi': 'ERROR',
                    'jenis': jenis_dokumen
                })

# Simpan Hasil
if ocr_results:
    df = pd.DataFrame(ocr_results)
    output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_paddle.csv'
    df.to_csv(output_csv_path, index=False)
    print(f"\nHasil OCR (PaddleOCR) disimpan di: {output_csv_path}")
    print(df.head())
else:
    print("Tidak ada data yang diproses.")

File augmentasi akan disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/

===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazahllm (Jenis: ijazah) =====


Memproses ijazahllm:   0%|          | 0/22 [00:00<?, ?it/s]C:\Users\ibuba\AppData\Local\Temp\ipykernel_18764\1773259101.py:16: DeprecationWarning: Please use `predict` instead.
  result = engine.ocr(img_array)
Memproses ijazahllm: 100%|██████████| 22/22 [24:30<00:00, 66.85s/it]



===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikatllm (Jenis: sertifikat) =====


Memproses sertifikatllm:  38%|███▊      | 6/16 [06:44<13:00, 78.04s/it]Resized image size (6889x9745) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (6889x9745) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (6889x9745) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (6889x9745) exceeds max_side_limit of 4000. Resizing to fit within limit.
Memproses sertifikatllm:  62%|██████▎   | 10/16 [16:01<11:41, 116.88s/it]Resized image size (5169x7309) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (5169x7309) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (5169x7309) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (5169x7309) exceeds max_side_limit of 4000. Resizing to fit within limit.
Memproses sertifikatllm: 100%|██████████| 16/16 [23:10<00:00, 86.91s/it] 



Hasil OCR (PaddleOCR) disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_paddle.csv
                                    nama_file_sumber  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  05ca0060115008c65dc3193f0ba4ecac_jpg.rf.4aec03...   

                                    nama_file_output  \
0  02081e1b-833a-494d-8e71-992671938014-160213173...   
1  02081e1b-833a-494d-8e71-992671938014-160213173...   
2  02081e1b-833a-494d-8e71-992671938014-160213173...   
3  02081e1b-833a-494d-8e71-992671938014-160213173...   
4  05ca0060115008c65dc3193f0ba4ecac_jpg.rf.4aec03...   

                       hasil_ocr    augmentasi   jenis  
0  n a o t o e e e e e e e i e e      original  ijazah  
1  n a o t o e e e e e e e i e e  aug_rotate_2  ijazah  
2  n a o t o e e e e e e e i e e    aug_blur_3

## LLM Qwen

In [12]:
# Path Folder Lokal
ijazah = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazahllm" 
sertifikat = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/sertifikatllm"
folders_to_process = [ijazah, sertifikat]
ocr_results = []
allowed_extensions = ('.pdf', '.png', '.jpg', '.jpeg')

# Folder Output Augmentasi
AUG_OUTPUT_DIR = "C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/" 
os.makedirs(AUG_OUTPUT_DIR, exist_ok=True)
print(f"File augmentasi akan disimpan di: {AUG_OUTPUT_DIR}")

for folder_path in folders_to_process:
    
    # --- Logic Jenis Dokumen ---
    jenis_dokumen = 'unknown'
    if folder_path == ijazah:
        jenis_dokumen = 'ijazah'
    elif folder_path == sertifikat:
        jenis_dokumen = 'sertifikat'

    print(f"\n===== Memulai proses di folder: {folder_path} (Jenis: {jenis_dokumen}) =====")
    
    if not os.path.exists(folder_path):
        print("Folder tidak ditemukan.")
        continue

    for filename in tqdm(os.listdir(folder_path), desc=f"Memproses {os.path.basename(folder_path)}"):
        if filename.lower().endswith(allowed_extensions):
            file_path = os.path.join(folder_path, filename)
            try:
                image = None
                
                # --- Load PDF/Image ---
                if filename.lower().endswith('.pdf'):
                    images_from_pdf = convert_from_path(
                        file_path, first_page=1, last_page=1, poppler_path=POPPLER_PATH
                    )
                    if images_from_pdf: image = images_from_pdf[0]
                else:
                    image = Image.open(file_path)
                
                if image:
                    # --- PROSES 1: OCR GAMBAR ASLI (PAKE LLM) ---
                    # Disini kita ganti pytesseract jadi run_lmstudio_ocr
                    raw_text_llm = run_lmstudio_ocr(image)
                    
                    # Cleaning ringan (spasi ganda)
                    clean_text_llm = re.sub(r'\s+', ' ', raw_text_llm).strip()
                    
                    ocr_results.append({
                        'nama_file_sumber': filename,
                        'nama_file_output': filename,
                        'hasil_ocr': clean_text_llm,
                        'augmentasi': 'original',
                        'jenis': jenis_dokumen 
                    })
                    
                    # --- PROSES 2: AUGMENTASI & OCR BARU (PAKE LLM) ---
                    base_name, _ = os.path.splitext(filename)
                    augmented_images = apply_augmentations(image)

                    for aug_name, aug_img in augmented_images.items():
                        
                        aug_filename = f"{base_name}_{aug_name}.jpg"
                        aug_filepath = os.path.join(AUG_OUTPUT_DIR, aug_filename)
                        
                        aug_img.save(aug_filepath, "JPEG")
                        
                        # OCR pada gambar augmentasi pake LLM juga
                        raw_text_aug = run_lmstudio_ocr(aug_img)
                        clean_text_aug = re.sub(r'\s+', ' ', raw_text_aug).strip()
                        
                        ocr_results.append({
                            'nama_file_sumber': filename, 
                            'nama_file_output': aug_filename, 
                            'hasil_ocr': clean_text_aug, 
                            'augmentasi': aug_name,
                            'jenis': jenis_dokumen
                        })

            except Exception as e:
                # print(f"\n!!! Gagal memproses file {filename}: {e} !!!")
                ocr_results.append({
                    'nama_file_sumber': filename,
                    'nama_file_output': 'ERROR',
                    'hasil_ocr': f'ERROR: {e}',
                    'augmentasi': 'ERROR',
                    'jenis': jenis_dokumen
                })

# Simpan Hasil
if ocr_results:
    df = pd.DataFrame(ocr_results)
    output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_lmstudio.csv'
    df.to_csv(output_csv_path, index=False)
    print(f"Hasil OCR LLM berhasil disimpan di: {output_csv_path}")
else:
    print("Tidak ada hasil OCR untuk disimpan.")

File augmentasi akan disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/augmented_data/

===== Memulai proses di folder: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/ijazahllm (Jenis: ijazah) =====


Memproses ijazahllm:  27%|██▋       | 6/22 [13:16<38:29, 144.35s/it]  

❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}
❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}


Memproses ijazahllm:  32%|███▏      | 7/22 [37:03<2:20:53, 563.56s/it]

❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}
❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}
❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}


...

❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}


Memproses sertifikatllm:  56%|█████▋    | 9/16 [40:50<59:52, 513.25s/it]

❌ Error Request: Error code: 400 - {'error': 'Reached context length of 8192 tokens, but this model does not currently support mid-generation context overflow because llama_memory_can_shift is 0. Try reloading with a larger context length or shortening the prompt/chat.'}


Memproses sertifikatllm: 100%|██████████| 16/16 [54:01<00:00, 202.60s/it]


Hasil OCR LLM berhasil disimpan di: C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_lmstudio.csv


In [19]:
# --- INPUT: Masukkan Path File Lo Di Sini ---
TARGET_FILE_PATH = r"C:/Users/ibuba/Kuliah/Proyek Akhir\Aplikasi/Dataset/ijazahllm/ijazah-transkrip-S2-Renovita.pdf"

# List Penampung Hasil
ocr_results = []

filename = os.path.basename(TARGET_FILE_PATH)
folder_path = os.path.dirname(TARGET_FILE_PATH)

# --- Otomatis tebak jenis dokumen dari nama folder ---
jenis_dokumen = 'unknown'
if 'ijazah' in folder_path.lower():
    jenis_dokumen = 'ijazah'
elif 'sertifikat' in folder_path.lower():
    jenis_dokumen = 'sertifikat'

print(f"\n===== Memproses File: {filename} (Jenis: {jenis_dokumen}) =====")

try:
    image = None
    
    # --- Load PDF/Image ---
    if filename.lower().endswith('.pdf'):
        print("   -> Mengonversi PDF...")
        images_from_pdf = convert_from_path(
            TARGET_FILE_PATH, first_page=1, last_page=1, poppler_path=POPPLER_PATH
        )
        if images_from_pdf: image = images_from_pdf[0]
    else:
        print("   -> Membuka Gambar...")
        image = Image.open(TARGET_FILE_PATH)
    
    if image:
        # --- PROSES 1: OCR GAMBAR ASLI ---
        print("   -> Menjalankan OCR pada gambar asli (LM Studio)...")
        raw_text_llm = run_lmstudio_ocr(image)
        clean_text_llm = re.sub(r'\s+', ' ', raw_text_llm).strip()
        
        ocr_results.append({
            'nama_file_sumber': filename,
            'nama_file_output': filename,
            'hasil_ocr': clean_text_llm,
            'augmentasi': 'original',
            'jenis': jenis_dokumen 
        })
        print("      ✅ OCR Asli Selesai.")

except Exception as e:
    print(f"\n!!! Gagal memproses file: {e} !!!")
    ocr_results.append({
        'nama_file_sumber': filename,
        'nama_file_output': 'ERROR',
        'hasil_ocr': f'ERROR: {e}',
        'augmentasi': 'ERROR',
        'jenis': jenis_dokumen
    })
    
# Simpan Hasil
if ocr_results:
    df = pd.DataFrame(ocr_results)
    
    # Simpan CSV dengan nama file spesifik
    base_name_clean = os.path.splitext(filename)[0]
    output_csv_path = f'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_{base_name_clean}.csv'
    
    df.to_csv(output_csv_path, index=False)
    print(f"\n✅ Selesai! Hasil OCR disimpan di:\n{output_csv_path}")
    
    # Preview
    print("\nPreview Hasil:")
    print(df[['nama_file_output', 'augmentasi', 'hasil_ocr']].head())
else:
    print("Tidak ada hasil OCR untuk disimpan.")


===== Memproses File: ijazah-transkrip-S2-Renovita.pdf (Jenis: ijazah) =====
   -> Mengonversi PDF...
   -> Menjalankan OCR pada gambar asli (LM Studio)...
      ✅ OCR Asli Selesai.

✅ Selesai! Hasil OCR disimpan di:
C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_ijazah-transkrip-S2-Renovita.csv

Preview Hasil:
                   nama_file_output augmentasi  \
0  ijazah-transkrip-S2-Renovita.pdf   original   

                                           hasil_ocr  
0  048/PENS-14/MIC/MT/RR/2020 No. : PIN : 5510120...  


## Cleaning

In [8]:
file_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_lmstudio.csv' 
try:
    df = pd.read_csv(file_path, sep=',') 
    print("File CSV berhasil dibaca. Kolom terdeteksi:")
    print(df.columns.tolist()) 
except Exception as e:
    print(f"Gagal membaca file: {e}")

def clean_ocr_text(text):
    text = str(text).lower() # Tambahkan str() untuk keamanan
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

if 'df' in locals():
    df['cleaned_ocr'] = df['hasil_ocr'].apply(clean_ocr_text)

    print("\nData setelah dibersihkan (Kolom 'cleaned_ocr'):")
    
    # Kolom 'jenis' akan otomatis terbawa
    print(df[['hasil_ocr', 'cleaned_ocr', 'jenis']].head()) 

    output_csv_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_clean_lmstudio.csv'
    
    # Saat disimpan, 'jenis' otomatis ikut tersimpan
    df.to_csv(output_csv_path, index=False) 
    print(f"File clean (termasuk kolom 'jenis') disimpan di {output_csv_path}")

File CSV berhasil dibaca. Kolom terdeteksi:
['nama_file_sumber', 'nama_file_output', 'hasil_ocr', 'augmentasi', 'jenis']

Data setelah dibersihkan (Kolom 'cleaned_ocr'):
                                           hasil_ocr  \
0  Nomor Seri : 0512/61201/2012 KEMENTERIAN PENDI...   
1  Nomor Seri : 0512/61201/2012 KEMENTERIAN PENDI...   
2  Nomor Seri : 0512/61201/2012 KEMENTERIAN PENDI...   
3  Nomor Seri : 0512/61201/2012 KEMENTERIAN PENDI...   
4  THE REGENTS OF THE University of California ON...   

                                         cleaned_ocr   jenis  
0  nomor seri 0512 61201 2012 kementerian pendidi...  ijazah  
1  nomor seri 0512 61201 2012 kementerian pendidi...  ijazah  
2  nomor seri 0512 61201 2012 kementerian pendidi...  ijazah  
3  nomor seri 0512 61201 2012 kementerian pendidi...  ijazah  
4  the regents of the university of california on...  ijazah  
File clean (termasuk kolom 'jenis') disimpan di C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_clean_lms

## Evaluasi Model OCR

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [3]:
# ==============================================================================
# 1. FUNGSI METRIK EVALUASI (CER & WER MANUAL)
# ==============================================================================
# Kita buat fungsi manual biar gak perlu install library tambahan yang ribet
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def preprocess_text_for_eval(text):
    """Membersihkan teks agar perbandingan adil (lowercase, hapus simbol non-alphanumeric)."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) # Hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip() # Normalisasi spasi
    return text

def preprocess_text_for_eval(text):
    """Membersihkan teks: lowercase, hapus simbol, normalisasi spasi."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_keyword_recall(ground_truth, ocr_result):
    """
    Menghitung Recall: (Kata GT yang ditemukan di OCR) / (Total Kata GT).
    Range: 0.0 - 1.0 (1.0 berarti Sempurna/Semua info ada).
    """
    def tokenize(text):
        return set(text.split())

    gt_tokens = tokenize(ground_truth)
    ocr_tokens = tokenize(ocr_result)
    
    if not gt_tokens: return 0.0 # Hindari pembagian nol
    
    # Cari kata yang COCOK (Irisan)
    matched_tokens = gt_tokens.intersection(ocr_tokens)
    
    # Hitung Recall
    recall_score = len(matched_tokens) / len(gt_tokens)
    return recall_score

In [3]:
file_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr/evaluasi model.xlsx'

try:
    df = pd.read_excel(file_path)
    print("✅ Data evaluasi berhasil dimuat.")
except Exception as e:
    print(f"❌ Gagal memuat data: {e}")
    raise e 

# List model yang akan dievaluasi
models = ['pytesseract', 'donut', 'keras', 'llm qwen']
ground_truth_col = 'teks asli'

# Cleaning Text
print("Membersihkan teks...")
for col in models + [ground_truth_col]:
    df[f'clean_{col}'] = df[col].fillna('').apply(preprocess_text_for_eval)

# Hitung Skor Per Baris (Global dulu)
print("Menghitung skor Keyword Recall...")
for model in models:
    df[f'{model}_RECALL'] = df.apply(
        lambda row: calculate_keyword_recall(row[f'clean_{ground_truth_col}'], row[f'clean_{model}']), 
        axis=1
    )

# ==============================================================================
# 3. FUNGSI PEMBUAT DASHBOARD (Modular)
# ==============================================================================
def generate_dashboard_per_jenis(df_subset, jenis_name):
    """
    Fungsi untuk membuat dashboard dan analisis spesifik per jenis dokumen.
    """
    print(f"\n--- Membuat Analisis untuk: {jenis_name.upper()} ---")
    
    # 1. Simpan CSV Spesifik
    output_csv = f'laporan_evaluasi_{jenis_name}.csv'
    df_subset.to_csv(output_csv, index=False)
    print(f"✅ Laporan CSV disimpan ke: {output_csv}")

    # 2. Setup Plot
    plt.style.use('seaborn-v0_8-whitegrid')
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(f'Dashboard Evaluasi OCR: {jenis_name.upper()} (Keyword Recall)', fontsize=20, fontweight='bold')

    # --- SUBPLOT 1: Overall Performance (Bar Chart) ---
    ax1 = fig.add_subplot(2, 2, 1)
    
    # Hitung rata-rata cuma buat subset ini
    avg_scores = df_subset[[f'{m}_RECALL' for m in models]].mean()
    recall_values = [avg_scores[f'{m}_RECALL'] for m in models]

    x = np.arange(len(models))
    bars = ax1.bar(x, recall_values, width=0.6, label='Keyword Recall', color='#2ecc71')

    # Label angka
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., 1.02*height,
                 f'{height*100:.1f}%', ha='center', va='bottom', fontweight='bold')

    ax1.set_ylabel('Recall Score')
    ax1.set_title(f'1. Performa Rata-rata ({jenis_name.title()})', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([m.upper() for m in models])
    ax1.set_ylim(0, 1.15)

    # --- SUBPLOT 2: Performance by Tipe File (Gambar vs PDF) ---
    ax2 = fig.add_subplot(2, 2, 2)
    # Group by 'tipe' hanya untuk data subset ini
    df_tipe = df_subset.groupby('tipe')[[f'{m}_RECALL' for m in models]].mean()
    
    if not df_tipe.empty:
        df_tipe.plot(kind='bar', ax=ax2, colormap='viridis')
        ax2.set_title(f'2. Analisis Tipe File pada {jenis_name.title()}', fontweight='bold')
        ax2.set_ylabel('Avg Recall Score')
        ax2.set_xlabel('Tipe Dokumen')
        ax2.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.set_ylim(0, 1.1)
    else:
        ax2.text(0.5, 0.5, "Data Tipe tidak tersedia", ha='center')

    # --- SUBPLOT 3: Distribusi Skor (Boxplot/Histogram ganti Bar horizontal model) ---
    # Karena ini spesifik jenis, kita ganti analisis "Jenis" (yg ada di kode lama) 
    # menjadi distribusi performa per model (Boxplot) untuk melihat kestabilan.
    ax3 = fig.add_subplot(2, 2, 3)
    data_boxplot = [df_subset[f'{m}_RECALL'] for m in models]
    ax3.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True)
    ax3.set_title(f'3. Stabilitas Model (Boxplot) - {jenis_name.title()}', fontweight='bold')
    ax3.set_ylabel('Recall Score Distribution')

    # --- SUBPLOT 4: Kesimpulan Teks ---
    ax4 = fig.add_subplot(2, 2, 4)
    ax4.axis('off')

    # Logika Penentuan Juara
    if recall_values:
        best_model_idx = np.argmax(recall_values)
        best_model_name = models[best_model_idx]
        best_model_score = recall_values[best_model_idx]
    else:
        best_model_name = "N/A"
        best_model_score = 0

    summary_text = f"""
    KESIMPULAN KHUSUS {jenis_name.upper()}:

    MODEL TERBAIK (Overall):
    {best_model_name.upper()} 
    (Keyword Recall: {best_model_score*100:.2f}%)

    ANALISIS BERDASARKAN TIPE FILE:
    """

    if not df_tipe.empty:
        for t in df_tipe.index:
            best_in_type = df_tipe.loc[t].idxmax().replace('_RECALL', '')
            score = df_tipe.loc[t].max()
            summary_text += f"- {t.title()}: Juara {best_in_type.upper()} ({score*100:.1f}% Recall)\n"
    else:
        summary_text += "- Tidak cukup data untuk analisis tipe file.\n"

    summary_text += "\n--------------------------------------------------\n"
    summary_text += f"Total Sampel {jenis_name.title()}: {len(df_subset)} Baris"

    ax4.text(0.05, 0.5, summary_text, fontsize=11, family='monospace', va='center')

    # Simpan Gambar
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    output_img = f'dashboard_evaluasi_{jenis_name}.png'
    plt.savefig(output_img, dpi=300)
    print(f"✅ Dashboard visual disimpan ke: {output_img}")
    plt.close(fig) # Tutup plot biar gak numpuk di memori

# ==============================================================================
# 4. EKSEKUSI PEMISAHAN (IJAZAH & SERTIFIKAT)
# ==============================================================================

# Pastikan kolom 'jenis' ada dan dinormalisasi (lowercase)
if 'jenis' in df.columns:
    df['jenis'] = df['jenis'].astype(str).str.lower().str.strip()
    
    # 1. Proses Ijazah
    df_ijazah = df[df['jenis'] == 'ijazah'].copy()
    if not df_ijazah.empty:
        generate_dashboard_per_jenis(df_ijazah, 'ijazah')
    else:
        print("⚠️ Tidak ada data dengan jenis 'ijazah'.")

    # 2. Proses Sertifikat
    df_sertifikat = df[df['jenis'] == 'sertifikat'].copy()
    if not df_sertifikat.empty:
        generate_dashboard_per_jenis(df_sertifikat, 'sertifikat')
    else:
        print("⚠️ Tidak ada data dengan jenis 'sertifikat'.")

else:
    print("❌ Kolom 'jenis' tidak ditemukan dalam dataset Excel.")

✅ Data evaluasi berhasil dimuat.
Membersihkan teks...
Menghitung skor Keyword Recall...

--- Membuat Analisis untuk: IJAZAH ---
✅ Laporan CSV disimpan ke: laporan_evaluasi_ijazah.csv


C:\Users\ibuba\AppData\Local\Temp\ipykernel_16180\3512680441.py:88: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax3.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True)


✅ Dashboard visual disimpan ke: dashboard_evaluasi_ijazah.png

--- Membuat Analisis untuk: SERTIFIKAT ---
✅ Laporan CSV disimpan ke: laporan_evaluasi_sertifikat.csv


C:\Users\ibuba\AppData\Local\Temp\ipykernel_16180\3512680441.py:88: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax3.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True)


✅ Dashboard visual disimpan ke: dashboard_evaluasi_sertifikat.png


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

# ==============================================================================
# 1. FUNGSI CLEANING & METRIK (Tetap sama)
# ==============================================================================
def preprocess_text_for_eval(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_keyword_recall(ground_truth, prediction):
    if not ground_truth: return 0.0
    gt_tokens = set(ground_truth.split())
    pred_tokens = set(prediction.split())
    if not gt_tokens: return 0.0
    matches = gt_tokens.intersection(pred_tokens)
    return len(matches) / len(gt_tokens)

# ==============================================================================
# 2. LOAD & PREP DATA
# ==============================================================================
file_path = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr/evaluasi model.xlsx'

try:
    df = pd.read_excel(file_path)
    print("✅ Data dimuat.")
except Exception as e:
    print(f"❌ Error: {e}"); exit()

models = ['pytesseract', 'donut', 'keras', 'llm qwen']
ground_truth_col = 'teks asli'

# Cleaning & Scoring
for col in models + [ground_truth_col]:
    df[f'clean_{col}'] = df[col].fillna('').apply(preprocess_text_for_eval)

for model in models:
    df[f'{model}_RECALL'] = df.apply(
        lambda row: calculate_keyword_recall(row[f'clean_{ground_truth_col}'], row[f'clean_{model}']), 
        axis=1
    )

# ==============================================================================
# 3. FUNGSI GENERATE PLOT TERPISAH
# ==============================================================================
def save_individual_plots(df_subset, jenis_name):
    print(f"\n--- Generasi Plot Terpisah untuk: {jenis_name.upper()} ---")
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # Warna Bar (Hijau segar)
    bar_color = '#2ecc71'
    
    # ---------------------------------------------------------
    # PLOT 1: Overall Performance (Rata-rata Recall)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6)) # Ukuran standar per gambar
    
    avg_scores = df_subset[[f'{m}_RECALL' for m in models]].mean()
    recall_values = [avg_scores[f'{m}_RECALL'] for m in models]
    x = np.arange(len(models))
    
    bars = plt.bar(x, recall_values, width=0.6, color=bar_color)
    
    # Label Angka
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., 1.01*height,
                 f'{height*100:.1f}%', ha='center', va='bottom', fontweight='bold')

    plt.title(f'Rata-rata Keyword Recall ({jenis_name.title()})', fontsize=14, fontweight='bold')
    plt.ylabel('Recall Score (0.0 - 1.0)')
    plt.xticks(x, [m.upper() for m in models])
    plt.ylim(0, 1.15)
    plt.tight_layout()
    
    filename1 = f'plot_{jenis_name}_1_overall.png'
    plt.savefig(filename1, dpi=300)
    print(f"✅ Disimpan: {filename1}")
    plt.close()

    # ---------------------------------------------------------
    # PLOT 2: Analisis Tipe File (Gambar vs PDF)
    # ---------------------------------------------------------
    df_tipe = df_subset.groupby('tipe')[[f'{m}_RECALL' for m in models]].mean()
    
    if not df_tipe.empty:
        plt.figure(figsize=(10, 6))
        # Plot Pandas langsung lebih gampang buat grouped bar
        ax = df_tipe.plot(kind='bar', colormap='viridis', figsize=(10, 6))
        
        plt.title(f'Performa Berdasarkan Tipe File ({jenis_name.title()})', fontsize=14, fontweight='bold')
        plt.ylabel('Avg Recall Score')
        plt.xlabel('Tipe Dokumen')
        plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.ylim(0, 1.1)
        plt.tight_layout()
        
        filename2 = f'plot_{jenis_name}_2_tipefile.png'
        plt.savefig(filename2, dpi=300)
        print(f"✅ Disimpan: {filename2}")
        plt.close()
    else:
        print("⚠️ Data tipe file tidak cukup untuk plotting.")

    # ---------------------------------------------------------
    # PLOT 3: Distribusi/Stabilitas (Boxplot)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    data_boxplot = [df_subset[f'{m}_RECALL'] for m in models]
    
    plt.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True,
                boxprops=dict(facecolor='#3498db', color='black'),
                medianprops=dict(color='yellow'))
    
    plt.title(f'Distribusi Stabilitas Model ({jenis_name.title()})', fontsize=14, fontweight='bold')
    plt.ylabel('Recall Score Distribution')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    filename3 = f'plot_{jenis_name}_3_boxplot.png'
    plt.savefig(filename3, dpi=300)
    print(f"✅ Disimpan: {filename3}")
    plt.close()

# ==============================================================================
# 4. EKSEKUSI
# ==============================================================================
if 'jenis' in df.columns:
    df['jenis'] = df['jenis'].astype(str).str.lower().str.strip()
    
    # Generate untuk Ijazah
    df_ijazah = df[df['jenis'] == 'ijazah']
    if not df_ijazah.empty:
        save_individual_plots(df_ijazah, 'ijazah')
        
    # Generate untuk Sertifikat
    df_sertifikat = df[df['jenis'] == 'sertifikat']
    if not df_sertifikat.empty:
        save_individual_plots(df_sertifikat, 'sertifikat')
else:
    print("Kolom 'jenis' tidak ditemukan.")

✅ Data dimuat.

--- Generasi Plot Terpisah untuk: IJAZAH ---
✅ Disimpan: plot_ijazah_1_overall.png
✅ Disimpan: plot_ijazah_2_tipefile.png


C:\Users\ibuba\AppData\Local\Temp\ipykernel_29212\2051195724.py:116: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True,


✅ Disimpan: plot_ijazah_3_boxplot.png

--- Generasi Plot Terpisah untuk: SERTIFIKAT ---
✅ Disimpan: plot_sertifikat_1_overall.png
✅ Disimpan: plot_sertifikat_2_tipefile.png


C:\Users\ibuba\AppData\Local\Temp\ipykernel_29212\2051195724.py:116: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_boxplot, labels=[m.upper() for m in models], patch_artist=True,


✅ Disimpan: plot_sertifikat_3_boxplot.png


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

# NER

## Library

In [5]:
import pandas as pd
import re
import os
from difflib import get_close_matches

## File Path

In [6]:
# File Input Utama
FILE_INPUT_OCR = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr/hasil_ocr_lmstudio.csv' 

# File Pendukung (Kamus)
FILE_UNIV = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/LIST UNIVERSITAS.csv'
FILE_JURUSAN = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/daftar_jurusan.xlsx'
FILE_LSP_BNSP = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/hasil_scraping_lsp_bnsp.csv'
FILE_SERTIF_INTER = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/hasil_scraping_sertifikasi_internasional_per_negara.csv'
FILE_SKILL = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/skill.csv'
FILE_BIDANG = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/bidang_keahlian.csv'
FILE_GELAR = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/daftar_gelar_fixed.csv'

# Output Files
OUTPUT_IJAZAH = 'hasil_ner_ijazah_final.csv'
OUTPUT_SERTIFIKAT = 'hasil_ner_sertifikat_final.csv'

In [22]:
# File Input Utama
FILE_INPUT_OCR = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_ijazah-transkrip-D4-Renovita.csv' 
# FILE_INPUT_OCR = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr_ijazah-transkrip-S2-Renovita.csv' 

# File Pendukung (Kamus)
FILE_UNIV = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/LIST UNIVERSITAS.csv'
FILE_JURUSAN = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/daftar_jurusan.xlsx'
FILE_LSP_BNSP = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/hasil_scraping_lsp_bnsp.csv'
FILE_SERTIF_INTER = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/hasil_scraping_sertifikasi_internasional_per_negara.csv'
FILE_SKILL = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/skill.csv'
FILE_BIDANG = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/bidang_keahlian.csv'
FILE_GELAR = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/file tambahan/daftar_gelar_fixed.csv'

# Output Files
OUTPUT_IJAZAH = 'hasil_oner_ijazah-transkrip-D4-Renovita.csv'
OUTPUT_SERTIFIKAT = 'hasil_ner_ijazah-transkrip-D4-Renovita.csv'

## Fungsi Cleaning dan Utility

In [7]:
def light_clean_for_ner(text):
    if pd.isna(text): return ""
    text = str(text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = text.replace('|', ' ').replace('©', '').replace('®', '').replace('~', '')
    text = re.sub(r'(\d)\s*[.,]\s*(\d)', r'\1.\2', text)
    text = re.sub(r'\s*:\s*', ': ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def fix_ocr_digit(text_angka):
    if not text_angka: return None
    text_angka = str(text_angka)
    replacements = {
        'l': '1', 'I': '1', 'i': '1', '|': '1', '!': '1',
        'O': '0', 'o': '0', 'D': '0',
        'S': '5', 's': '5', 'B': '8', 'Z': '2', 'g': '9', 'b': '6'
    }
    for char, digit in replacements.items():
        text_angka = text_angka.replace(char, digit)
    text_angka = re.sub(r'[^\d\.,]', '', text_angka)
    return text_angka

def konversi_kata_ke_angka(teks_angka):
    if not teks_angka: return None
    teks_angka = teks_angka.lower().replace('-', ' ').strip()
    nilai_kata = {
        'nol': 0, 'satu': 1, 'se': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5, 'enam': 6, 'tujuh': 7, 'delapan': 8, 'sembilan': 9,
        'sepuluh': 10, 'sebelas': 11, 'seratus': 100, 'seribu': 1000,
        'zero': 0, 'one': 1, 'a': 1, 'an': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9,
        'ten': 10, 'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19,
        'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50, 'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90
    }
    words = teks_angka.split()
    total_value = 0; current_value = 0
    for word in words:
        if word in ['and', 'of', 'dan']: continue
        if word in nilai_kata: current_value += nilai_kata[word]
        elif word == 'belas': current_value += 10
        elif word == 'puluh':
            if current_value == 0: current_value = 1
            current_value *= 10; total_value += current_value; current_value = 0
        elif word in ['ratus', 'hundred']:
            if current_value == 0: current_value = 1
            current_value *= 100
        elif word in ['ribu', 'thousand']:
            if current_value == 0: current_value = 1
            current_value *= 1000; total_value += current_value; current_value = 0
    total_value += current_value
    return total_value

def muat_kamus_data():
    print("Sedang memuat kamus data...")
    kamus = {}
    
    def load_safe(path, col_name):
        if os.path.exists(path):
            try:
                if path.endswith('.xlsx'): df = pd.read_excel(path)
                else: df = pd.read_csv(path, on_bad_lines='skip')
                return set(df[col_name].dropna().astype(str).str.lower().str.strip())
            except Exception as e:
                print(f"⚠️ Warning: Gagal baca {path} (Error: {e})")
                return set()
        return set()

    try:
        # Kamus Ijazah
        kamus['univ'] = load_safe(FILE_UNIV, 'nama')
        kamus['jurusan'] = load_safe(FILE_JURUSAN, 'Jurusan')
        if os.path.exists(FILE_GELAR):
            try:
                df_gelar = pd.read_csv(FILE_GELAR, on_bad_lines='skip')
                raw_gelar = df_gelar['Gelar'].dropna().astype(str).unique()
                valid_gelar = set()
                blacklist_csv = ['S1', 'S2', 'S3', 'D1', 'D2', 'D3', 'D4', 'SP1', 'SP2', 'PROFESI']
                for g in raw_gelar:
                    g_clean = g.strip()
                    if len(g_clean) < 3 or g_clean.upper() in blacklist_csv: continue
                    if '.' in g_clean or (g_clean.isupper() and len(g_clean) >= 3): valid_gelar.add(g_clean) 
                kamus['gelar_baku'] = valid_gelar
                print(f"✅ Kamus Gelar dimuat: {len(valid_gelar)} gelar valid.")
            except Exception as e:
                print(f"❌ Error load gelar: {e}"); kamus['gelar_baku'] = set()
        else: kamus['gelar_baku'] = set()

        # Kamus Sertifikat
        ls_penerbit = set()
        ls_penerbit.update(load_safe(FILE_LSP_BNSP, 'Nama LSP'))
        ls_penerbit.update(load_safe(FILE_SERTIF_INTER, 'Nama Lembaga'))
        kamus['penerbit'] = ls_penerbit

        ls_skill = set()
        ls_skill.update(load_safe(FILE_SKILL, 'skill'))
        ls_skill.update(load_safe(FILE_BIDANG, 'bidang_keahlian'))
        kamus['skill'] = ls_skill

        print(f"✅ Semua kamus berhasil dimuat.")
        return kamus

    except Exception as e:
        print(f"❌ Error Fatal saat memuat kamus: {e}"); return None

## NER Ijazah

In [8]:
def proses_ner_ijazah(row, kamus):
    teks = str(row['cleaned_for_ner'])
    teks_safe = " " + teks + " " 
    hasil = {}

    # 1. NIM (Fix)
    pola_nim_standar = r'(?:nim|nirm|npm|nomor induk|stb|nomor register|student\'s number|registration number)\s*[:.]?\s*([0-9lIOBsS]{7,25})'
    match_nim = re.search(pola_nim_standar, teks_safe, re.IGNORECASE)
    if match_nim:
        hasil['NIM'] = fix_ocr_digit(match_nim.group(1))
    else:
        # Lapis 2: Pola "Nama : NIM"
        potensi = re.finditer(r'([a-z\s\.,]{4,50})\s*[:]\s*([0-9]{8,18})', teks_safe, re.I)
        kandidat_nim = None
        blacklist = ['tanggal', 'date', 'lahir', 'birth', 'sk', 'nomor', 'nik', 'ipk', 'lulus', 'tahun']
        for m in potensi:
            if not any(x in m.group(1).lower() for x in blacklist):
                kandidat_nim = m.group(2); break
        if kandidat_nim: hasil['NIM'] = fix_ocr_digit(kandidat_nim)
        else:
            fb = re.search(r'\b([0-9]{9,12})\b', teks_safe); hasil['NIM'] = fix_ocr_digit(fb.group(1)) if fb else None

    # 2. JURUSAN
    pola_jurusan_label = r'(?:jurusan|program studi|prodi|study program|major)\s*[:]\s*([a-zA-Z\s,&-]+?)(?=\s+(?:program|status|fakultas|lulus|tanggal|jenjang|\n|$))'
    match_jurusan = re.search(pola_jurusan_label, teks_safe, re.IGNORECASE)
    
    if match_jurusan:
        hasil['Jurusan'] = match_jurusan.group(1).strip().title()
    else:
        jurusan_match = [j for j in kamus['jurusan'] if j in teks.lower()]
        hasil['Jurusan'] = max(jurusan_match, key=len).title() if jurusan_match else None

    # 3. GELAR
    kandidat_gelar = []
    pola_konteks = r'(?:memakai gelar|menyandang gelar|diberikan gelar|memperoleh gelar|sebutan|degree of|academic degree)\s+(?:akademik\s+)?([a-zA-Z\s\.\(\)\-]+?)(?=\s+(?:The\b|This\b|Has\b|With\b|beserta\b|hak\b|dan\b|kewajiban\b|dengan\b|sesuai\b|\.))'
    match_konteks = re.search(pola_konteks, teks_safe, re.IGNORECASE)
    gelar_konteks = None
    if match_konteks:
        raw_konteks = match_konteks.group(1).strip()
        raw_konteks = re.sub(r'^(akademik|kesarjanaan)\s+', '', raw_konteks, flags=re.I)
        if raw_konteks.endswith('.'): raw_konteks = raw_konteks[:-1]
        if len(raw_konteks) > 2 and len(raw_konteks) < 60: gelar_konteks = raw_konteks
    
    pola_singkatan = r'\b([A-Z][a-z]{0,3}\.(?:[A-Z][a-z]{0,3}\.?)+)\b'
    kandidat_gelar.extend(re.findall(pola_singkatan, teks))
    if 'gelar_baku' in kamus:
        for g in kamus['gelar_baku']:
            if re.search(r'\b' + re.escape(g) + r'\b', teks): kandidat_gelar.append(g)

    gelar_final = []
    blacklist_umum = ['P.T.', 'C.V.', 'U.U.', 'P.P.', 'No.', 'S.K.', 'St.', 'Jl.', 'N.I.P.', 'S.Pd', 'S-2', 'S-1'] 
    zona_bahaya = []
    for m in re.finditer(r'(rektor|dekan|ketua|direktur|nip|nidn)', teks_safe, re.I):
        zona_bahaya.append((m.start(), m.end() + 80))

    semua_kandidat = set(kandidat_gelar)
    if gelar_konteks: semua_kandidat.add(gelar_konteks)

    for g in semua_kandidat:
        g_clean = g.strip()
        if len(g_clean) < 3 or (len(g_clean) > 50 and g_clean != gelar_konteks): continue
        if g_clean.upper() in blacklist_umum: continue
        is_safe = True
        if g_clean != gelar_konteks:
            for m_g in re.finditer(re.escape(g_clean), teks_safe):
                g_start = m_g.start()
                for b_start, b_end in zona_bahaya:
                    if b_start <= g_start <= b_end: is_safe = False; break
                if not is_safe: break
        if is_safe: gelar_final.append(g_clean)

    if gelar_konteks: hasil['Gelar'] = gelar_konteks
    else: hasil['Gelar'] = ", ".join(list(set(gelar_final))) if gelar_final else None

    # 4. IPK
    match_ipk = re.search(r'(?:ipk|indeks prestasi|kumulatif|gpa|grade point)\s*[:.]?\s*([2-3]\.\d{2}|4\.00)', teks_safe, re.I)
    hasil['IPK'] = match_ipk.group(1) if match_ipk else None

    # 5. TAHUN
    list_tahun = []
    pola_tahun_angka = r'\b((?:19|20)\d{2})\b'
    for t in re.findall(pola_tahun_angka, teks_safe):
        t_int = int(t)
        if 1950 <= t_int <= 2030: list_tahun.append(t_int)
    
    pola_huruf = r'(?:tahun|year|dated)\s+((?:dua\s*ribu|seribu|two\s*thousand|nineteen|twenty)\s+(?:[\w\s\-]+?)(?=\s|$))'
    match_huruf = re.finditer(pola_huruf, teks_safe, re.IGNORECASE)
    for m in match_huruf:
        if 'ribu' in m.group(1).lower() or 'thousand' in m.group(1).lower():
            angka = konversi_kata_ke_angka(m.group(1).strip())
            if angka and 1950 <= angka <= 2030: list_tahun.append(angka)
    
    if list_tahun: hasil['Tahun_Lulus'] = str(max(list_tahun))
    else: hasil['Tahun_Lulus'] = None

    # 6. PREDIKAT
    list_pred = [r'summa\s*cum\s*laude', r'magna\s*cum\s*laude', r'cum\s*laude', r'dengan\s*pujian', r'sangat\s*memuaskan', r'memuaskan', r'baik\s*sekali']
    match_pred = re.search(r'\b(' + '|'.join(list_pred) + r')\b', teks_safe, re.I)
    hasil['Predikat'] = match_pred.group(1).title() if match_pred else None

    # 7. UNIVERSITAS
    univ_match = [u for u in kamus['univ'] if u in teks.lower()]
    hasil['Universitas'] = max(univ_match, key=len).title() if univ_match else None

    return hasil

## NER Sertifikat

In [9]:
def proses_ner_sertifikat(row, kamus):
    teks = str(row['cleaned_for_ner'])
    teks_safe = " " + teks + " " 
    teks_lower = teks.lower()
    hasil = {}

    # Fallback Judul Panjang (jika Pola 1 gagal)
    if not hasil.get('Judul_Sertifikat'):
         pola_fallback_panjang = r'(?:sertifikat|certificate)\s+of\s+([a-zA-Z\s,]+?)(?=\s+menyatakan|\s+presented\s+to|\s+diberikan\s+kepada|\s+this\s+certifies)'
         match_fb = re.search(pola_fallback_panjang, teks, re.I)
         if match_fb:
             hasil['Judul_Sertifikat'] = match_fb.group(1).strip().title()
         else:
             hasil['Judul_Sertifikat'] = None

    # 2. Lembaga Penerbit (TWEAKED: Ambil Nama Terpanjang)
    penerbit_match = [p for p in kamus['penerbit'] if p in teks_lower]
    
    if penerbit_match:
        hasil['Lembaga_Penerbit'] = max(penerbit_match, key=len).title()
    else:
        # FALLBACK 2: Cari Nama Universitas
        univ_match = [u for u in kamus['univ'] if u in teks_lower]
        if univ_match:
             hasil['Lembaga_Penerbit'] = max(univ_match, key=len).title()
        else:
             hasil['Lembaga_Penerbit'] = "Lainnya"

    # 3. Skill / Kompetensi (REVISED: Boundary Lebih Luas)
    skills_found = []
    if kamus['skill']:
        for s in kamus['skill']:
            if re.search(r'\b' + re.escape(s) + r'\b', teks_lower):
                skills_found.append(s.title())
                
    hasil['Detected_Skills'] = ", ".join(sorted(list(set(skills_found)))) if skills_found else None

    # 4. Tahun Sertifikat
    list_tahun = []
    pola_tahun_angka = r'\b((?:19|20)\d{2})\b' 
    for t in re.findall(pola_tahun_angka, teks_safe):
        t_int = int(t)
        if 1950 <= t_int <= 2030: list_tahun.append(t_int)
    
    pola_huruf = r'(?:tahun|year|dated)\s+((?:dua\s*ribu|seribu|two\s*thousand|nineteen|twenty)\s+(?:[\w\s\-]+?)(?=\s|$))'
    match_huruf = re.finditer(pola_huruf, teks_safe, re.IGNORECASE)
    for m in match_huruf:
        if 'ribu' in m.group(1).lower() or 'thousand' in m.group(1).lower():
            angka = konversi_kata_ke_angka(m.group(1).strip())
            if angka and 1950 <= angka <= 2030: list_tahun.append(angka)
    
    if list_tahun:
        hasil['Tahun_Sertifikat'] = str(max(list_tahun))
    else:
        match_date_en = re.search(r'[A-Z][a-z]+\s+\d{1,2},\s*((?:19|20)\d{2})', teks_safe)
        hasil['Tahun_Sertifikat'] = match_date_en.group(1) if match_date_en else None
        
    return hasil

## Hasil

In [10]:
try:
    if not os.path.exists(FILE_INPUT_OCR): raise FileNotFoundError(f"File {FILE_INPUT_OCR} tidak ditemukan.")
    df_utama = pd.read_csv(FILE_INPUT_OCR)
    col_text = 'hasil_ocr' if 'hasil_ocr' in df_utama.columns else 'ocr_raw_text'
    if col_text not in df_utama.columns: col_text = df_utama.columns[2]
    
    print(f"✅ Data OCR dimuat: {len(df_utama)} baris.")
    print("Melakukan Light Cleaning...")
    df_utama['cleaned_for_ner'] = df_utama[col_text].apply(light_clean_for_ner)

except Exception as e:
    print(f"❌ Gagal memuat data OCR: {e}"); exit()

kamus_data = muat_kamus_data()

if kamus_data:
    print("\nMemproses data...")
    
    # --- Ijazah ---
    df_ijazah = df_utama[df_utama['jenis'] == 'ijazah'].copy()
    if not df_ijazah.empty:
        print(f"-> Mengolah {len(df_ijazah)} dokumen Ijazah...")
        
        hasil_list = []
        for _, row in df_ijazah.iterrows():
            hasil_list.append(proses_ner_ijazah(row, kamus_data))
            
        df_hasil_ner = pd.DataFrame(hasil_list)
        
        df_ijazah = df_ijazah.reset_index(drop=True)
        df_hasil_ner = df_hasil_ner.reset_index(drop=True)
        
        cols_id = ['nama_file_sumber', 'jenis', 'augmentasi'] if 'augmentasi' in df_ijazah.columns else ['nama_file_sumber', 'jenis']
        
        df_final_ijazah = pd.concat([df_ijazah[cols_id], df_hasil_ner], axis=1)
        df_final_ijazah.to_csv(OUTPUT_IJAZAH, index=False)
        print(f"   ✅ Ijazah Selesai: {OUTPUT_IJAZAH}")
        print("   Preview Columns:", df_final_ijazah.columns.tolist())
    
    # --- Sertifikat ---
    df_sertifikat = df_utama[df_utama['jenis'] == 'sertifikat'].copy()
    if not df_sertifikat.empty:
        print(f"-> Mengolah {len(df_sertifikat)} dokumen Sertifikat...")
        
        hasil_list = []
        for _, row in df_sertifikat.iterrows():
            hasil_list.append(proses_ner_sertifikat(row, kamus_data))
            
        df_hasil_ner = pd.DataFrame(hasil_list)
        
        df_sertifikat = df_sertifikat.reset_index(drop=True)
        df_hasil_ner = df_hasil_ner.reset_index(drop=True)
        
        cols_id = ['nama_file_sumber', 'jenis', 'augmentasi'] if 'augmentasi' in df_sertifikat.columns else ['nama_file_sumber', 'jenis']
        
        df_final_sertif = pd.concat([df_sertifikat[cols_id], df_hasil_ner], axis=1)
        df_final_sertif.to_csv(OUTPUT_SERTIFIKAT, index=False)
        print(f"   ✅ Sertifikat Selesai: {OUTPUT_SERTIFIKAT}")
        print("   Preview Columns:", df_final_sertif.columns.tolist())

    print("\n=== SELESAI ===")

✅ Data OCR dimuat: 152 baris.
Melakukan Light Cleaning...
Sedang memuat kamus data...
✅ Kamus Gelar dimuat: 235 gelar valid.
✅ Semua kamus berhasil dimuat.

Memproses data...
-> Mengolah 88 dokumen Ijazah...
   ✅ Ijazah Selesai: hasil_ner_ijazah_final.csv
   Preview Columns: ['nama_file_sumber', 'jenis', 'augmentasi', 'NIM', 'Jurusan', 'Gelar', 'IPK', 'Tahun_Lulus', 'Predikat', 'Universitas']
-> Mengolah 64 dokumen Sertifikat...
   ✅ Sertifikat Selesai: hasil_ner_sertifikat_final.csv
   Preview Columns: ['nama_file_sumber', 'jenis', 'augmentasi', 'Judul_Sertifikat', 'Lembaga_Penerbit', 'Detected_Skills', 'Tahun_Sertifikat']

=== SELESAI ===


# NER LLM

In [1]:
import pandas as pd
import json
import os
from openai import OpenAI
from tqdm import tqdm
import re

In [2]:
# 1. KONFIGURASI
LM_STUDIO_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL_ID = "qwen/qwen3-vl-4b" 

# File Input
FILE_INPUT = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ocr/hasil_ocr_lmstudio.csv'

# File Output (DIPISAH)
OUTPUT_IJAZAH = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ner_qwen_ijazah_final.csv'
OUTPUT_SERTIFIKAT = 'C:/Users/ibuba/Kuliah/Proyek Akhir/Aplikasi/Dataset/hasil_ner_qwen_sertifikat_final.csv'

In [3]:
# 2. DEFINISI PROMPT (OTAKNYA)
def get_prompt(jenis):
    if jenis == 'ijazah':
        return """
        Lakukan OCR dan Ekstraksi Entitas (NER) dari gambar Ijazah ini.
        Output WAJIB JSON murni dengan keys:
        - "OCR_Raw": (Semua teks yang terbaca)
        - "NIM": Nomor Induk Mahasiswa atau Nomor Registrasi (Cari angka panjang, jangan NIK/No Ijazah).
        - "Jurusan": Program Studi.
        - "Gelar": Gelar akademik (Contoh: S.Kom, Ph.D, S. T, Sarjana Teknik, Sarjana Komputer, DLL).
        - "Tahun_Lulus": Tahun kelulusan (4 digit, jangan buat dalam bentuk desimal).
        - "Universitas": Nama Perguruan Tinggi.
        
        Aturan:
        1. Jika data tidak ditemukan, isi dengan teks "Tidak Ditemukan Data". 
        2. Jangan mengarang data, akan tetapi jika entitas terdapat typo atau kesalahan penulisan maka perbaiki kesalahannya.
        3. Hanya berikan JSON murni, tanpa teks pembuka/penutup/markdown.
        """
    elif jenis == 'sertifikat':
        return """
        Lakukan OCR dan Ekstraksi Entitas (NER) dari gambar Sertifikat ini.
        Output WAJIB JSON murni dengan keys:
        - "OCR_Raw": (Semua teks yang terbaca)
        - "Judul_Sertifikat": Nama pelatihan/event utama.
        - "id_sertifikat" : Nomor sertifikat jika ada.
        - "Lembaga_Penerbit": Organisasi penerbit.
        - "Skill": Daftar skill/topik utama (pisahkan koma).
        - "Tahun_Sertifikat": Tahun terbit (4 digit, jangan buat dalam bentuk desimal).
        - "Masa_Berlaku": Tahun kadaluarsa (4 digit, jangan buat dalam bentuk desimal, isi "Tidak Ditemukan Data" jika seumur hidup).
        
        Aturan:
        1. Jika data tidak ditemukan, isi dengan teks "Tidak Ditemukan Data". 
        2. Jangan mengarang data, akan tetapi jika entitas terdapat typo atau kesalahan penulisan maka perbaiki kesalahannya.
        3. Hanya berikan JSON murni, tanpa teks pembuka/penutup/markdown.
        """
    return None

# 3. FUNGSI API LLM
def run_ner_llm(client, text, jenis):
    system_prompt = get_prompt(jenis)
    if not system_prompt: return {}

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Teks OCR:\n{text}"}
            ],
            temperature=0.1, max_tokens=500
        )
        
        raw_content = response.choices[0].message.content
        clean_json = raw_content.strip().replace("```json", "").replace("```", "")
        return json.loads(clean_json)

    except Exception:
        return {}

In [4]:
# 4. EKSEKUSI UTAMA
# A. Koneksi
try:
    client = OpenAI(base_url=LM_STUDIO_URL, api_key=API_KEY)
    client.models.list()
except:
    print("❌ Gagal koneksi ke LM Studio."); exit()

# B. Load Data
if not os.path.exists(FILE_INPUT):
    print("❌ File input tidak ditemukan."); exit()

df = pd.read_csv(FILE_INPUT)
print(f"📊 Memproses {len(df)} baris data...")

# Cek nama kolom teks
col_text = 'hasil_ocr' if 'hasil_ocr' in df.columns else df.columns[2]

# Container Hasil
list_hasil_ijazah = []
list_hasil_sertif = []

# C. Loop Processing
for index, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    text = str(row[col_text])
    jenis = row['jenis']
    
    # Skip kalau teks kosong/error
    if len(text) < 10 or "ERROR" in text: continue

    # Eksekusi NER
    ner_result = run_ner_llm(client, text, jenis)
    
    # Gabung metadata asli (nama file, augmentasi, dll) + hasil NER
    combined_data = row.to_dict()
    combined_data.update(ner_result)

    # Pisahkan ke keranjang masing-masing
    if jenis == 'ijazah':
        list_hasil_ijazah.append(combined_data)
    elif jenis == 'sertifikat':
        list_hasil_sertif.append(combined_data)

# D. Simpan Output Terpisah
print("\n💾 Menyimpan Hasil...")

# --- SAVE IJAZAH ---
if list_hasil_ijazah:
    df_ijazah = pd.DataFrame(list_hasil_ijazah)
    # Rapikan kolom (taruh entitas penting di depan)
    cols_target = ['NIM', 'Jurusan', 'Gelar', 'Tahun_Lulus', 'Universitas', 'IPK']
    final_cols = [c for c in df_ijazah.columns if c not in cols_target] + cols_target # Metadata dulu baru NER
    # (Opsional: Kalau mau NER dulu baru Metadata, tukar posisi list di atas)
    
    df_ijazah.to_csv(OUTPUT_IJAZAH, index=False)
    print(f"✅ Ijazah ({len(df_ijazah)} data) -> {os.path.basename(OUTPUT_IJAZAH)}")
else:
    print("⚠️ Tidak ada data Ijazah yang berhasil diekstrak.")

# --- SAVE SERTIFIKAT ---
if list_hasil_sertif:
    df_sertif = pd.DataFrame(list_hasil_sertif)
    cols_target = ['Judul_Sertifikat', 'Lembaga_Penerbit', 'Skill', 'Tahun_Sertifikat', 'Masa_Berlaku']
    
    df_sertif.to_csv(OUTPUT_SERTIFIKAT, index=False)
    print(f"✅ Sertifikat ({len(df_sertif)} data) -> {os.path.basename(OUTPUT_SERTIFIKAT)}")
else:
    print("⚠️ Tidak ada data Sertifikat yang berhasil diekstrak.")

📊 Memproses 152 baris data...


Extracting: 100%|██████████| 152/152 [45:12<00:00, 17.84s/it]


💾 Menyimpan Hasil...
✅ Ijazah (80 data) -> hasil_ner_qwen_ijazah_final.csv
✅ Sertifikat (62 data) -> hasil_ner_qwen_sertifikat_final.csv


In [8]:
from thefuzz import fuzz

coba = fuzz.token_set_ratio("Muhammad Rafly Gunawan".lower(), "Muhammad ihsan A".lower())
coba

67